# Cell-type prediction from methylation `.cz` (`CellTypeClassifier`)

Predict the cell type of a shallow single-cell (e.g. MiSeq snm3C-seq, ~200 reads/cell) `.cz` file using deep-sequencing per-cell-type pseudobulk `.cz` files.

**Method** — a transparent, training-free naive-Bayes / methylation-frequency deconvolution:

1. For each candidate type `t` and cytosine `c`, estimate a *continuous* methylation frequency from the pseudobulk with Beta shrinkage:
   $$\theta_{c,t} = \frac{m_{c,t} + \alpha_0}{n_{c,t} + \alpha_0 + \beta_0}$$
2. Score a query cell by the aggregated per-cytosine Bernoulli log-likelihood, summed in log-space:
   $$\log P(t\mid\text{cell}) \propto \log\pi_t + \lambda_{CG}\!\sum_{c\in CG}[\,s_c\log\theta^{CG}_{c,t} + (1-s_c)\log(1-\theta^{CG}_{c,t})\,] + \lambda_{CH}\!\sum_{c\in CH}[\cdots]$$
3. Softmax over types → calibrated probabilities (supports abstention).

CpG and CpH are modelled as two independent channels; the split is read from the `context` column of a `build_ref` **reference** `.cz` (the cell-type / query files usually store only `mc`/`cov`).

> All inputs (query + pseudobulks) must be aligned to the same reference axis (one row per reference cytosine, in reference order).

In [ ]:
import os, struct, tempfile
import numpy as np
import cytozip as czip
from cytozip import CellTypeClassifier

## 1. Build a small self-contained example

Three cell types (`Oligo` / `Astro` / `Micro`) over 200 cytosines (alternating CpG / CpH contexts). The cell-type `.cz` store only `mc` / `cov` (as real per-cell / pseudobulk files do), and a separate **reference** `.cz` (`pos`, `strand`, `context`, from `czip build_ref`) supplies the CpG/CpH context. All files share one axis. In real use these come from `czip allc2cz --reference` (per cell) and `czip merge_cz` / pseudobulk (per cell type).

In [ ]:
np.random.seed(0)
d = tempfile.mkdtemp()
S = 200
contexts = [b'CGN' if i % 2 == 0 else b'CAC' for i in range(S)]   # half CpG, half CpH
cg = np.array([i for i in range(S) if i % 2 == 0])
ch = np.array([i for i in range(S) if i % 2 == 1])

# per-type methylation frequency profiles (continuous, discriminative)
prof = {}
f = np.empty(S); f[cg] = 0.90; f[ch] = np.where(np.arange(len(ch)) < len(ch)//2, 0.10, 0.02); prof['Oligo'] = f.copy()
f = np.empty(S); f[cg] = 0.60; f[ch] = np.where(np.arange(len(ch)) < len(ch)//2, 0.02, 0.10); prof['Astro'] = f.copy()
f = np.empty(S); f[cg] = 0.75; f[ch] = 0.05;                                                 prof['Micro'] = f.copy()

def write_cz(path, frac, depth):
    """Write an mc/cov-only .cz (as real per-cell / pseudobulk files are)."""
    cov = np.maximum(np.random.poisson(depth, S), 0).astype(np.uint16)
    mc = np.random.binomial(cov, frac).astype(np.uint16)
    w = czip.Writer(path, formats=['H', 'H'], columns=['mc', 'cov'],
                    chunk_dims=['chrom'], sort_col=False)
    w.write_chunk(b''.join(struct.pack('<HH', int(mc[i]), int(cov[i]))
                           for i in range(S)), ['chr1'])
    w.close()

# a build_ref-style reference .cz carrying pos/strand/context (the CpG/CpH axis)
reference = os.path.join(d, 'reference.cz')
w = czip.Writer(reference, formats=['Q', 'c', '3s'], columns=['pos', 'strand', 'context'],
                chunk_dims=['chrom'], sort_col='pos', delta_cols=['pos'])
w.write_chunk(b''.join(struct.pack('<Qc3s', i + 1, b'+', contexts[i]) for i in range(S)), ['chr1'])
w.close()

# deep pseudobulk cell-type files (mc/cov-only, high coverage)
pseudobulks = {}
for t in prof:
    p = os.path.join(d, f'{t}.cz'); write_cz(p, prof[t], depth=60); pseudobulks[t] = p

# a shallow query cell that is truly Oligo (~3x coverage)
query = os.path.join(d, 'cellQ.cz'); write_cz(query, prof['Oligo'], depth=3)
pseudobulks

## 2. Fit the classifier

`fit` estimates the CpG/CpH frequencies once. The cell-type files are `mc`/`cov`-only, so the CpG/CpH split comes from the `reference` `.cz`. Optional `top_cg` / `top_ch` keep only the most discriminative sites; `cell_counts` enables an abundance / tempered prior at predict time.

In [ ]:
clf = CellTypeClassifier(lambda_cg=1.0, lambda_ch=1.0).fit(
    pseudobulks=pseudobulks,
    reference=reference,                                      # build_ref .cz supplying context
    cell_counts={'Oligo': 1000, 'Astro': 500, 'Micro': 50},   # optional (for abundance prior)
    top_cg=None, top_ch=None,                                 # or e.g. 0.1 to keep top 10% sites
)

## 3. Predict a single cell

`predict` returns the label, confidence, the full softmax `proba` Series, and the unnormalized `log_posterior`. `prior_alpha` tempers the abundance prior (`0` = uniform, `1` = full abundance).

In [ ]:
res = clf.predict(query, prior_alpha=0.0)          # uniform prior
print('label     :', res['label'])
print('confidence: %.4f' % res['confidence'])
res['proba'].round(4)

In [ ]:
# abstain when the top probability is not confident enough
clf.predict(query, abstain_threshold=0.999)['label']

## 4. Save the trained model, then predict a new cell after reloading

`save` writes a single `.npz` (hyper-parameters + reference axis + pre-computed `log_theta`). `load` restores it with `allow_pickle=False` — no refitting and **no need to re-read the pseudobulks**, so prediction on new cells is instant.

In [ ]:
model_path = os.path.join(d, 'celltype_model.npz')
clf.save(model_path)
print('model size: %.1f KB' % (os.path.getsize(model_path) / 1024))

clf2 = CellTypeClassifier.load(model_path)
res2 = clf2.predict(query, prior_alpha=1.0)        # abundance prior this time
print('reloaded label:', res2['label'], '(conf %.4f)' % res2['confidence'])

## 5. Batch-predict many cells

`predict_batch` accepts a `{cell_id: path}` dict (or a list of paths) and returns a labels DataFrame plus a cell x cell_type probability matrix.

In [ ]:
labels, proba = clf2.predict_batch({'cellQ': query})
display(labels)
proba.round(4)

## 6. Applying it to real data

The cell-type pseudobulk and single-cell `.cz` normally store only `mc`/`cov` and are aligned to the same reference axis (produced by `czip build_ref` + `czip allc2cz --reference`). Pass the `build_ref` reference `.cz` via `reference=` so the classifier can split CpG vs CpH:

```python
from cytozip import CellTypeClassifier

pseudobulks = {
    'Oligo': 'pseudobulk/Oligo.cz',
    'Astro': 'pseudobulk/Astro.cz',
    'Micro': 'pseudobulk/Micro.cz',
    'OPC':   'pseudobulk/OPC.cz',
}

clf = CellTypeClassifier(lambda_cg=1.0, lambda_ch=1.0).fit(
    pseudobulks=pseudobulks,
    reference='mm10_with_chrL.allc.cz',   # build_ref .cz supplying the per-row context
    top_cg=0.1, top_ch=0.1,               # keep the top 10% most discriminative sites
)
clf.save('bg_celltype_model.npz')

# later / elsewhere — predict shallow MiSeq cells directly
clf = CellTypeClassifier.load('bg_celltype_model.npz')
labels, proba = clf.predict_batch({
    cell_id: f'miseq/{cell_id}.cz' for cell_id in cell_ids
}, prior_alpha=0.0, abstain_threshold=0.6)
```

**Tuning tips**: start with a uniform prior (`prior_alpha=0`) to check raw discriminative power, then sweep `prior_alpha` / `top_*` / `lambda_ch` on a held-out set while watching **macro-F1** (fair to rare types) and the confusion matrix, not just overall accuracy.